# Import Required Libraries
Import necessary libraries such as transformers, torch, matplotlib, and plotly for model loading, inference, and visualization.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re

# Load Gemma 4 Model
Load the smallest Gemma 4 model (e.g., 2B or equivalent) using Hugging Face transformers, configuring for both Thinking and Standard modes.

In [6]:
# Load a free model as placeholder (GPT-2) - Replace with Gemma 4 when available
model_name = "gpt2"  # Free model for demo; replace with "google/gemma-4-2b-it" when Gemma 4 is released
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="auto")

# Function to generate response
def generate_response(prompt, thinking=False, max_length=512):
    if thinking:
        # Simulate thinking mode with a prompt
        prompt = f"Think step by step: {prompt}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=max_length, do_sample=True, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

# Prepare Test Cases
Define a set of complex logic puzzles and code debugging tasks as input prompts for benchmarking.

In [ ]:
# Define test cases
test_cases = [
    {
        "type": "logic_puzzle",
        "prompt": "There are 12 balls, one of which is heavier or lighter. Using a balance scale, find the odd ball in 3 weighings.",
        "description": "Classic balance puzzle"
    },
    {
        "type": "code_debug",
        "prompt": "Debug this Python function: def factorial(n): if n == 0: return 1 else: return n * factorial(n-1). It works for positive n but fails for negative. Fix it.",
        "description": "Factorial function debug"
    },
    {
        "type": "logic_puzzle",
        "prompt": "A lily pad doubles in size every day. It covers the pond in 30 days. When was the pond half covered?",
        "description": "Lily pad growth puzzle"
    }
]

# Run Inferences in Thinking Mode
Execute the model in Thinking mode on the test cases, capturing intermediate reasoning steps and final outputs.

In [ ]:
# Run thinking mode inferences
thinking_results = []
for case in test_cases:
    prompt = case["prompt"]
    response = generate_response(prompt, thinking=True)
    # Extract thinking part (simulate by splitting on "Answer:")
    if "Answer:" in response:
        thinking = response.split("Answer:")[0].replace("Think step by step: ", "").strip()
        final_answer = response.split("Answer:")[1].strip()
    else:
        thinking = "No thinking captured"
        final_answer = response
    thinking_results.append({
        "case": case["description"],
        "thinking": thinking,
        "final_answer": final_answer
    })
    print(f"Thinking for {case['description']}: {thinking[:100]}...")

# Run Inferences in Standard Mode
Execute the model in Standard mode on the same test cases, capturing direct outputs for comparison.

In [ ]:
# Run standard mode inferences
standard_results = []
for case in test_cases:
    prompt = case["prompt"]
    response = generate_response(prompt, thinking=False)
    standard_results.append({
        "case": case["description"],
        "answer": response
    })
    print(f"Standard for {case['description']}: {response[:100]}...")

# Collect and Compare Outputs
Parse and compare the outputs from both modes, identifying corrections or differences in reasoning paths.

In [ ]:
# Compare outputs
comparisons = []
for think, std in zip(thinking_results, standard_results):
    # Simple comparison: check if answers differ
    diff = think["final_answer"] != std["answer"]
    corrections = len(re.findall(r'correct|wrong|mistake', think["thinking"].lower()))
    comparisons.append({
        "case": think["case"],
        "thinking_answer": think["final_answer"],
        "standard_answer": std["answer"],
        "differ": diff,
        "corrections_in_thinking": corrections
    })

# Display comparisons
import pandas as pd
df = pd.DataFrame(comparisons)
print(df)

# Visualize Results Dashboard
Create interactive visualizations (e.g., side-by-side text comparisons, graphs of reasoning steps, and correction highlights) using Plotly or similar.

In [ ]:
# Visualize results
fig = make_subplots(rows=1, cols=2, subplot_titles=("Differences in Answers", "Corrections in Thinking"))

# Bar chart for differences
fig.add_trace(go.Bar(x=df["case"], y=df["differ"].astype(int), name="Answers Differ"), row=1, col=1)

# Bar chart for corrections
fig.add_trace(go.Bar(x=df["case"], y=df["corrections_in_thinking"], name="Corrections Count"), row=1, col=2)

fig.update_layout(title_text="Thinking vs Standard Mode Comparison")
fig.show()

# Side-by-side text comparison (print for now)
for comp in comparisons:
    print(f"\nCase: {comp['case']}")
    print(f"Standard: {comp['standard_answer'][:200]}...")
    print(f"Thinking: {comp['thinking_answer'][:200]}...")
    print(f"Thinking Trace: {thinking_results[comparisons.index(comp)]['thinking'][:300]}...")